# Step 4 & 6: 1D ResNet Architecture + Training with Adam

Train the full 1D ResNet (~1.2M params) with Adam optimizer.

Hyperparameters (paper defaults):
- α=0.001, β1=0.9, β2=0.999, ε=1e-8
- Batch=128, Max epochs=100, Early stopping patience=15
- CosineAnnealingLR, Dropout=0.4, weight_decay=1e-4
- FocalLoss (γ=2, α_t=1/class_freq)

Target: ≥98% accuracy, ≥93% macro F1

In [ ]:
import sys; sys.path.insert(0, '..')
import torch
from src.utils.seed import set_seed
from src.utils.device import get_device
from src.models.resnet1d import ResNet1D
from src.training.focal_loss import FocalLoss, compute_class_frequencies
from src.training.trainer import Trainer
from src.training.callbacks import EarlyStopping, ModelCheckpoint
from src.optimizers.optimizer_factory import build_optimizer

In [ ]:
set_seed(42)
device = get_device()
model = ResNet1D(num_classes=5, dropout=0.4).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
optimizer = build_optimizer(model, 'adam', weight_decay=1e-4)
class_freqs = compute_class_frequencies(splits['y_train'])
focal_loss = FocalLoss(gamma=2.0, class_frequencies=class_freqs)

trainer = Trainer(
    model, optimizer, focal_loss, device,
    callbacks=[EarlyStopping(patience=15), ModelCheckpoint(optimizer_name='adam')],
    use_lr_scheduler=True,
    max_epochs=100,
)
history = trainer.fit(train_loader, val_loader)